# 00 - Capability check

Run once (interactively) on the Free Edition workspace to confirm what the pipeline can rely on:

| Check | Why it matters |
|---|---|
| Spark session | Every pipeline task needs it |
| Landing volume readable | Source for Auto Loader |
| PyPI packages (`langdetect`, `model2vec`) | Language detection + small CPU embedding model |
| Hugging Face reachable | Decides whether the model is downloaded directly or uploaded to a volume |
| Foundation Model / `ai_query` | Optional free LLM fallback for skills |

Nothing is written by this notebook.

## Before running: add dependencies in the Environment panel (do NOT use `%pip`)

On serverless, `%pip install` builds a notebook environment that Spark must download; on Free Edition this can fail with `ENVIRONMENT_DOWNLOAD_USER_ERROR.NOT_FOUND`.

1. Open the notebook, click the **Environment** icon in the right side panel.
2. Under **Dependencies** add two lines: `langdetect==1.0.9` and `model2vec`.
3. Click **Apply** (the session restarts), then **Run all**.

If Spark still fails, click **Connect → Reset session** (or detach/reattach) and run again.

In [ ]:
dbutils.widgets.text("landing_path", "/Volumes/jobseeker/default/scrapes", "Landing volume path")
dbutils.widgets.text("embedding_model", "minishlab/potion-base-8M", "Embedding model (Hugging Face id)")

LANDING_PATH = dbutils.widgets.get("landing_path").rstrip("/")
EMBEDDING_MODEL = dbutils.widgets.get("embedding_model")

results = {}


def record(check, ok, detail=""):
    results[check] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)[:300]}
    print(f"[{'PASS' if ok else 'FAIL'}] {check}: {detail}")

## Environment

In [ ]:
try:
    current = spark.sql("SELECT current_catalog() AS catalog, current_user() AS user").first()
    record("spark_session", True, f"catalog={current.catalog}, user={current.user}, spark={spark.version}")
except Exception as exc:
    message = str(exc)
    hint = " -> remove %pip, add deps in the Environment panel, then Reset session" if "ENVIRONMENT_DOWNLOAD" in message else ""
    record("spark_session", False, message.splitlines()[0] + hint)

## Landing volume

In [ ]:
try:
    folders = dbutils.fs.ls(LANDING_PATH)
    csv_files = []
    for folder in folders:
        if folder.isDir():
            csv_files.extend(f for f in dbutils.fs.ls(folder.path) if f.name.endswith(".csv"))
        elif folder.name.endswith(".csv"):
            csv_files.append(folder)

    by_source = {}
    for file_info in csv_files:
        prefix = file_info.name.split("_", 1)[0]
        by_source[prefix] = by_source.get(prefix, 0) + 1

    record("landing_volume", True, f"{len(folders)} entries, {len(csv_files)} csv files, by source={by_source}")
except Exception as exc:
    record("landing_volume", False, exc)

## Python packages (PyPI)

In [ ]:
try:
    from langdetect import DetectorFactory, detect_langs

    DetectorFactory.seed = 0
    samples = {
        "en": "We are looking for a senior data engineer with PySpark experience.",
        "fr": "Nous recherchons un analyste marketing pour rejoindre notre équipe à Paris.",
        "es": "Buscamos un perfil proactivo con experiencia en medios y manejo de Excel.",
    }
    detected = {expected: str(detect_langs(text)[0]) for expected, text in samples.items()}
    record("langdetect", True, detected)
except Exception as exc:
    record("langdetect", False, exc)

try:
    import model2vec  # noqa: F401

    record("model2vec_import", True, getattr(model2vec, "__version__", "installed"))
except Exception as exc:
    record("model2vec_import", False, exc)

try:
    import torch

    record("torch_preinstalled", True, torch.__version__)
except Exception as exc:
    record("torch_preinstalled", False, f"not available ({exc.__class__.__name__}) - fine, model2vec does not need torch")

## Hugging Face access + model load

In [ ]:
import requests

try:
    response = requests.get(f"https://huggingface.co/api/models/{EMBEDDING_MODEL}", timeout=15)
    record("huggingface_reachable", response.status_code == 200, f"HTTP {response.status_code}")
except Exception as exc:
    record("huggingface_reachable", False, exc)

if results.get("huggingface_reachable", {}).get("status") == "PASS":
    try:
        import numpy as np
        from model2vec import StaticModel

        model = StaticModel.from_pretrained(EMBEDDING_MODEL)
        vectors = model.encode(["Senior Data Engineer PySpark Databricks", "Data Engineering pipelines ETL", "Payroll HR operations"])
        vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
        similar = float(vectors[0] @ vectors[1])
        dissimilar = float(vectors[0] @ vectors[2])
        record("embedding_model_load", similar > dissimilar, f"dim={vectors.shape[1]}, similar={similar:.3f}, dissimilar={dissimilar:.3f}")
    except Exception as exc:
        record("embedding_model_load", False, exc)
else:
    print("Hugging Face blocked: the model will be uploaded to /Volumes/jobseeker/ops/pipeline/models/ instead.")

## Foundation Model APIs / `ai_query` (optional)

In [ ]:
chat_endpoint = None
try:
    from databricks.sdk import WorkspaceClient

    workspace = WorkspaceClient()
    endpoints = [endpoint.name for endpoint in workspace.serving_endpoints.list() if endpoint.name.startswith("databricks-")]
    record("serving_endpoints", bool(endpoints), endpoints[:15])
    preferred = [name for name in endpoints if any(key in name for key in ("llama", "gpt-oss", "gemma", "claude", "qwen"))]
    chat_endpoint = (preferred or endpoints or [None])[0]
except Exception as exc:
    record("serving_endpoints", False, exc)

if chat_endpoint:
    try:
        reply = spark.sql(f"SELECT ai_query('{chat_endpoint}', 'Reply with the single word OK') AS reply").first().reply
        record("ai_query", True, f"{chat_endpoint}: {reply!r}")
    except Exception as exc:
        record("ai_query", False, f"{chat_endpoint}: {exc}")

## Summary

In [ ]:
# Plain-text summary first so it prints even when the Spark session is broken
width = max(len(check) for check in results)
for check, value in results.items():
    print(f"{value['status']:<5} {check:<{width}}  {value['detail']}")

try:
    summary = [(check, value["status"], value["detail"]) for check, value in results.items()]
    display(spark.createDataFrame(summary, "check STRING, status STRING, detail STRING"))
except Exception as exc:
    print(f"(table display skipped, Spark unavailable: {str(exc).splitlines()[0]})")